# Stage 01 Only: Speaker Diarization

Notebook này chỉ chạy đúng một file audio bạn tự nhập đường dẫn, in trực tiếp kết quả speaker diarization, không đi tiếp sang music clean, overlap, ASR hay HTML viewer.

In [ ]:
from pathlib import Path

# Sửa dòng này cho từng audio bạn muốn test.
# Ví dụ: AUDIO_INPUT_PATH = '/kaggle/input/my-audio-folder/sample.wav'
AUDIO_INPUT_PATH = '/kaggle/input/YOUR_DATASET/YOUR_AUDIO.wav'

PROJECT_ROOT = Path('/kaggle/working/sommelier')
PIPELINE_DIR = PROJECT_ROOT / 'podcast-pipeline'
CONFIG_PATH = PIPELINE_DIR / 'config.json'
OUTPUT_ROOT = PROJECT_ROOT / 'sommelier_batch_outputs' / 'diarization_only_runs'

AUDIO_PATH = Path(AUDIO_INPUT_PATH)
if not AUDIO_INPUT_PATH or 'YOUR_AUDIO' in AUDIO_INPUT_PATH or not AUDIO_PATH.exists():
    raise FileNotFoundError(f'Hãy sửa AUDIO_INPUT_PATH thành đường dẫn audio thật trên Kaggle: {AUDIO_INPUT_PATH}')

RUN_DIR = OUTPUT_ROOT / f'run_full_{AUDIO_PATH.stem}'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('AUDIO_INPUT_PATH =', AUDIO_PATH)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RUN_DIR =', RUN_DIR)

In [ ]:
!python {PIPELINE_DIR / 'run_stage_diarization_only.py'} \
  --input_audio_path "{AUDIO_PATH}" \
  --config_path "{CONFIG_PATH}" \
  --output_root "{OUTPUT_ROOT}" \
  --vad \
  --speaker-link-threshold 0.75 \
  --diar_device_index 0 \
  --sortformer_device_index 0

In [ ]:
import json
import pandas as pd

DIARIZATION_JSON = RUN_DIR / '01_diarization' / 'diarization.json'
if not DIARIZATION_JSON.exists():
    raise FileNotFoundError(f'Không tìm thấy diarization output: {DIARIZATION_JSON}')

payload = json.loads(DIARIZATION_JSON.read_text(encoding='utf-8'))
segments = payload.get('segments', [])
metadata = payload.get('metadata', {})

print('diarization_json =', DIARIZATION_JSON)
print('audio_duration_seconds =', metadata.get('audio_duration_seconds'))
print('processing_time_seconds =', metadata.get('processing_time_seconds'))
print('rt_factor =', metadata.get('rt_factor'))
print('segment_count =', len(segments))

df = pd.DataFrame(segments)
if df.empty:
    print('Không có segment diarization nào.')
else:
    df['duration'] = (df['end'].astype(float) - df['start'].astype(float)).round(3)
    display(df[['index', 'start', 'end', 'duration', 'speaker']])

    speaker_summary = (
        df.groupby('speaker', dropna=False)
        .agg(segments=('speaker', 'size'), total_seconds=('duration', 'sum'))
        .reset_index()
        .sort_values('speaker')
    )
    speaker_summary['total_seconds'] = speaker_summary['total_seconds'].round(3)
    display(speaker_summary)